# 09 - Gold | Tendências Digitais

## Objetivo

Consolidar os sinais de relevância digital observados no YouTube
Trending e no Google Trends durante 2025.

## Fontes

- YouTube Trending Brasil
- Google Trends Brasil

## Estratégia

### YouTube

A presença no Trending será analisada principalmente por:

- quantidade de vídeos únicos;
- quantidade de aparições no Trending;
- número de dias distintos em que cada vídeo apareceu;
- participação de cada categoria no total de vídeos únicos.

As métricas de visualizações, curtidas e comentários são snapshots
acumulados. Portanto, não serão somadas entre diferentes dias do
mesmo vídeo.

Para métricas de engajamento será utilizado o último snapshot
disponível de cada vídeo.

### Google Trends

Serão calculados por categoria:

- índice médio;
- índice máximo;
- índice mínimo;
- mediana;
- quantidade de semanas observadas.

## Escopos de comparação

### Núcleo multifuente

Categorias presentes em CETIC + YouTube + Google Trends:

- Notícias
- Esportes
- Música
- Humor
- Games

### Comparação ampliada CETIC + YouTube

Além do núcleo:

- Animações
- Tutoriais / Educação
- Influenciadores

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_youtube = spark.table(
    "workspace.mvp_silver.youtube_trending_br_2025"
)

df_trends = spark.table(
    "workspace.mvp_silver.google_trends_2025"
)

print("YouTube Silver:", df_youtube.count())
print("Google Trends Silver:", df_trends.count())

In [0]:
campos_youtube = [
    "video_id",
    "data_trending",
    "categoria",
    "categoria_original",
    "nivel_comparabilidade",
    "categoria_comparavel",
    "views",
    "likes",
    "comments"
]

campos_trends = [
    "data_semana",
    "categoria",
    "indice_trends"
]

print("=== YOUTUBE ===")

for campo in campos_youtube:
    print(
        campo,
        "->",
        "OK" if campo in df_youtube.columns
        else "NÃO ENCONTRADO"
    )

print("\n=== GOOGLE TRENDS ===")

for campo in campos_trends:
    print(
        campo,
        "->",
        "OK" if campo in df_trends.columns
        else "NÃO ENCONTRADO"
    )

In [0]:
df_dias_video = (
    df_youtube
    .groupBy(
        "video_id"
    )
    .agg(
        F.countDistinct(
            "data_trending"
        ).alias(
            "dias_trending"
        )
    )
)

In [0]:
display(
    df_dias_video
    .orderBy(
        F.desc("dias_trending")
    )
    .limit(20)
)

In [0]:
janela_ultimo_snapshot = (
    Window
    .partitionBy("video_id")
    .orderBy(
        F.col("data_trending").desc()
    )
)

In [0]:
df_youtube_ultimo = (
    df_youtube
    .withColumn(
        "_ordem_snapshot",
        F.row_number().over(
            janela_ultimo_snapshot
        )
    )
    .filter(
        F.col("_ordem_snapshot") == 1
    )
    .drop(
        "_ordem_snapshot"
    )
)

In [0]:
print(
    "Vídeos no último snapshot:",
    df_youtube_ultimo.count()
)

print(
    "Vídeos únicos na Silver:",
    df_youtube
    .select("video_id")
    .distinct()
    .count()
)

In [0]:
df_youtube_video = (
    df_youtube_ultimo
    .join(
        df_dias_video,
        on="video_id",
        how="left"
    )
)

In [0]:
display(
    df_youtube_video
    .select(
        "video_id",
        "categoria",
        "nivel_comparabilidade",
        "data_trending",
        "dias_trending",
        "views",
        "likes",
        "comments"
    )
    .limit(20)
)

In [0]:
total_videos_youtube = (
    df_youtube_video.count()
)

print(
    "Total de vídeos únicos:",
    total_videos_youtube
)

In [0]:
df_aparicoes_categoria = (
    df_youtube
    .groupBy(
        "categoria"
    )
    .agg(
        F.count("*").alias(
            "aparicoes_trending"
        )
    )
)

In [0]:
df_youtube_categoria = (
    df_youtube_video
    .groupBy(
        "categoria",
        "nivel_comparabilidade",
        "categoria_comparavel"
    )
    .agg(
        F.count("*").alias(
            "videos_unicos"
        ),

        F.avg(
            "dias_trending"
        ).alias(
            "media_dias_trending"
        ),

        F.max(
            "dias_trending"
        ).alias(
            "max_dias_trending"
        ),

        F.avg(
            "views"
        ).alias(
            "media_views_ultimo_snapshot"
        ),

        F.expr(
            "percentile_approx(views, 0.5)"
        ).alias(
            "mediana_views_ultimo_snapshot"
        ),

        F.avg(
            "likes"
        ).alias(
            "media_likes_ultimo_snapshot"
        ),

        F.expr(
            "percentile_approx(likes, 0.5)"
        ).alias(
            "mediana_likes_ultimo_snapshot"
        ),

        F.avg(
            "comments"
        ).alias(
            "media_comments_ultimo_snapshot"
        ),

        F.expr(
            "percentile_approx(comments, 0.5)"
        ).alias(
            "mediana_comments_ultimo_snapshot"
        )
    )
)

In [0]:
df_youtube_gold = (
    df_youtube_categoria
    .join(
        df_aparicoes_categoria,
        on="categoria",
        how="left"
    )
)

In [0]:
df_youtube_gold = (
    df_youtube_gold
    .withColumn(
        "participacao_videos_pct",
        F.round(
            (
                F.col("videos_unicos")
                /
                F.lit(total_videos_youtube)
            ) * 100,
            2
        )
    )
)

In [0]:
total_aparicoes_youtube = (
    df_youtube.count()
)

print(
    "Total de aparições:",
    total_aparicoes_youtube
)

In [0]:
df_youtube_gold = (
    df_youtube_gold
    .withColumn(
        "participacao_aparicoes_pct",
        F.round(
            (
                F.col("aparicoes_trending")
                /
                F.lit(total_aparicoes_youtube)
            ) * 100,
            2
        )
    )
)

In [0]:
categorias_nucleo = [
    "Notícias",
    "Esportes",
    "Música",
    "Humor",
    "Games"
]

categorias_ampliadas = [
    "Animações",
    "Tutoriais / Educação",
    "Influenciadores"
]

df_youtube_gold = (
    df_youtube_gold
    .withColumn(
        "escopo_comparacao",
        F.when(
            F.col("categoria").isin(
                categorias_nucleo
            ),
            "Núcleo - CETIC + YouTube + Google Trends"
        )
        .when(
            F.col("categoria").isin(
                categorias_ampliadas
            ),
            "Ampliado - CETIC + YouTube"
        )
        .otherwise(
            "Fora do cruzamento principal"
        )
    )
)

In [0]:
df_youtube_gold = (
    df_youtube_gold
    .withColumn(
        "ano_referencia",
        F.lit(2025)
    )
    .withColumn(
        "fonte",
        F.lit(
            "YouTube Trending Brasil"
        )
    )
    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
df_youtube_gold = (
    df_youtube_gold

    .withColumn(
        "media_dias_trending",
        F.round(
            "media_dias_trending",
            2
        )
    )

    .withColumn(
        "media_views_ultimo_snapshot",
        F.round(
            "media_views_ultimo_snapshot",
            2
        )
    )

    .withColumn(
        "media_likes_ultimo_snapshot",
        F.round(
            "media_likes_ultimo_snapshot",
            2
        )
    )

    .withColumn(
        "media_comments_ultimo_snapshot",
        F.round(
            "media_comments_ultimo_snapshot",
            2
        )
    )
)

In [0]:
display(
    df_youtube_gold
    .select(
        "categoria",
        "nivel_comparabilidade",
        "escopo_comparacao",
        "videos_unicos",
        "participacao_videos_pct",
        "aparicoes_trending",
        "participacao_aparicoes_pct",
        "media_dias_trending",
        "mediana_views_ultimo_snapshot"
    )
    .orderBy(
        F.desc("videos_unicos")
    )
)

In [0]:
soma_videos_categorias = (
    df_youtube_gold
    .agg(
        F.sum(
            "videos_unicos"
        ).alias("total")
    )
    .first()["total"]
)

print(
    "Vídeos únicos originais:",
    total_videos_youtube
)

print(
    "Soma dos vídeos por categoria:",
    soma_videos_categorias
)

print(
    "Consistente:",
    total_videos_youtube
    == soma_videos_categorias
)

In [0]:
janela_ranking_youtube = (
    Window
    .orderBy(
        F.desc(
            "participacao_videos_pct"
        )
    )
)

df_youtube_gold = (
    df_youtube_gold
    .withColumn(
        "ranking_youtube",
        F.dense_rank().over(
            janela_ranking_youtube
        )
    )
)

In [0]:
(
    df_youtube_gold.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "workspace.mvp_gold.tendencias_youtube_2025"
    )
)

## Google Trends

O Google Trends utiliza um índice relativo de 0 a 100.

O valor 100 representa o maior nível de interesse observado dentro
da consulta configurada, e não 100% das buscas realizadas.

Por isso, o índice será tratado como uma medida relativa de interesse,
sem interpretação como participação percentual absoluta.

In [0]:
display(
    df_trends
    .select(
        "data_semana",
        "categoria",
        "indice_trends"
    )
    .orderBy(
        "data_semana",
        "categoria"
    )
    .limit(30)
)

In [0]:
df_trends_gold = (
    df_trends
    .groupBy(
        "categoria"
    )
    .agg(
        F.countDistinct(
            "data_semana"
        ).alias(
            "semanas_observadas"
        ),

        F.round(
            F.avg(
                "indice_trends"
            ),
            2
        ).alias(
            "indice_medio"
        ),

        F.min(
            "indice_trends"
        ).alias(
            "indice_minimo"
        ),

        F.max(
            "indice_trends"
        ).alias(
            "indice_maximo"
        ),

        F.expr(
            "percentile_approx(indice_trends, 0.5)"
        ).alias(
            "indice_mediano"
        )
    )
)

In [0]:
janela_ranking_trends = (
    Window
    .orderBy(
        F.desc(
            "indice_medio"
        )
    )
)

df_trends_gold = (
    df_trends_gold
    .withColumn(
        "ranking_google_trends",
        F.dense_rank().over(
            janela_ranking_trends
        )
    )
)

In [0]:
df_trends_gold = (
    df_trends_gold
    .withColumn(
        "ano_referencia",
        F.lit(2025)
    )
    .withColumn(
        "fonte",
        F.lit(
            "Google Trends Brasil"
        )
    )
    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
display(
    df_trends_gold
    .orderBy(
        "ranking_google_trends"
    )
)

In [0]:
print(
    "Categorias Google Trends:",
    df_trends_gold.count()
)

fora_dominio = (
    df_trends_gold
    .filter(
        (F.col("indice_medio") < 0)
        |
        (F.col("indice_medio") > 100)
    )
    .count()
)

print(
    "Índices médios fora de 0-100:",
    fora_dominio
)

In [0]:
(
    df_trends_gold.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "workspace.mvp_gold.interesse_google_2025"
    )
)

In [0]:
df_youtube_nucleo = (
    df_youtube_gold
    .filter(
        F.col("categoria").isin(
            categorias_nucleo
        )
    )
    .select(
        "categoria",
        "videos_unicos",
        "participacao_videos_pct",
        "aparicoes_trending",
        "participacao_aparicoes_pct",
        "media_dias_trending",
        "ranking_youtube"
    )
)

In [0]:
df_trends_nucleo = (
    df_trends_gold
    .select(
        "categoria",
        "indice_medio",
        "indice_mediano",
        "indice_maximo",
        "ranking_google_trends"
    )
)

In [0]:
df_tendencias_nucleo = (
    df_youtube_nucleo
    .join(
        df_trends_nucleo,
        on="categoria",
        how="inner"
    )
)

In [0]:
print(
    "Categorias no núcleo:",
    df_tendencias_nucleo.count()
)

display(
    df_tendencias_nucleo
    .orderBy(
        "categoria"
    )
)

In [0]:
max_youtube = (
    df_tendencias_nucleo
    .agg(
        F.max(
            "participacao_videos_pct"
        ).alias("maximo")
    )
    .first()["maximo"]
)

df_tendencias_nucleo = (
    df_tendencias_nucleo
    .withColumn(
        "score_youtube_0_100",
        F.round(
            (
                F.col("participacao_videos_pct")
                /
                F.lit(max_youtube)
            ) * 100,
            2
        )
    )
)

In [0]:
max_trends = (
    df_tendencias_nucleo
    .agg(
        F.max(
            "indice_medio"
        ).alias("maximo")
    )
    .first()["maximo"]
)

df_tendencias_nucleo = (
    df_tendencias_nucleo
    .withColumn(
        "score_google_0_100",
        F.round(
            (
                F.col("indice_medio")
                /
                F.lit(max_trends)
            ) * 100,
            2
        )
    )
)

In [0]:
df_tendencias_nucleo = (
    df_tendencias_nucleo
    .withColumn(
        "score_tendencia_digital",
        F.round(
            (
                F.col("score_youtube_0_100")
                +
                F.col("score_google_0_100")
            ) / 2,
            2
        )
    )
)

In [0]:
janela_ranking_digital = (
    Window
    .orderBy(
        F.desc(
            "score_tendencia_digital"
        )
    )
)

df_tendencias_nucleo = (
    df_tendencias_nucleo
    .withColumn(
        "ranking_tendencia_digital",
        F.dense_rank().over(
            janela_ranking_digital
        )
    )
)

In [0]:
df_tendencias_nucleo = (
    df_tendencias_nucleo
    .withColumn(
        "ano_referencia",
        F.lit(2025)
    )
    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
display(
    df_tendencias_nucleo
    .select(
        "categoria",
        "participacao_videos_pct",
        "indice_medio",
        "score_youtube_0_100",
        "score_google_0_100",
        "score_tendencia_digital",
        "ranking_tendencia_digital"
    )
    .orderBy(
        "ranking_tendencia_digital"
    )
)

In [0]:
(
    df_tendencias_nucleo.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "workspace.mvp_gold.tendencias_digitais_nucleo_2025"
    )
)

In [0]:
%sql

SHOW TABLES IN workspace.mvp_gold;

In [0]:
print(
    "Gold CETIC:",
    spark.table(
        "workspace.mvp_gold.perfil_geracional_cetic_2025"
    ).count()
)

print(
    "Gold YouTube:",
    spark.table(
        "workspace.mvp_gold.tendencias_youtube_2025"
    ).count()
)

print(
    "Gold Google Trends:",
    spark.table(
        "workspace.mvp_gold.interesse_google_2025"
    ).count()
)

print(
    "Gold núcleo digital:",
    spark.table(
        "workspace.mvp_gold.tendencias_digitais_nucleo_2025"
    ).count()
)

## Resultado da Gold - Tendências Digitais

A camada Gold consolidou os sinais observados no YouTube Trending
e no Google Trends durante 2025.

### YouTube

A análise utilizou vídeos únicos e aparições no Trending como
indicadores principais de relevância.

As métricas acumuladas de visualizações, curtidas e comentários
não foram somadas entre diferentes datas, evitando dupla contagem
de snapshots do mesmo vídeo.

Para análises de engajamento foi considerado o último snapshot
disponível de cada vídeo.

### Google Trends

Os índices semanais foram agregados por categoria.

O índice do Google Trends representa interesse relativo dentro da
consulta realizada e não deve ser interpretado como percentual
absoluto de buscas.

### Núcleo multifuente

Foram harmonizadas cinco categorias presentes simultaneamente em
CETIC, YouTube e Google Trends:

- Notícias
- Esportes
- Música
- Humor
- Games

Os sinais de YouTube e Google Trends foram normalizados para uma
escala comparativa de 0 a 100.

Foi criado o indicador `score_tendencia_digital`, calculado como
média simples entre os dois sinais normalizados.

Esse indicador representa força digital relativa entre as categorias
analisadas e não percentual de consumo da população.

Tabelas criadas:

`workspace.mvp_gold.tendencias_youtube_2025`

`workspace.mvp_gold.interesse_google_2025`

`workspace.mvp_gold.tendencias_digitais_nucleo_2025`